# HS071 Optimization Problem

For a solution to the HS071 constrained function minimization problem using a Python script instead of a notebook, see the [HS071 Python script documentation](../scripts/hs071.rst).

## Description

HS071 is Problem 71 from the collection of nonlinear programming test problems by Hock and Schittkowski <cite data-footcite="Hock:1981">(1981)</cite>, who cite Bartholomew-Biggs <cite data-footcite="Bartholomew-Biggs:1976">(1976)</cite> as the original source. The problem is to minimize the function
$$
   f(x)=x_{0} x_{3}\left(x_{0}+x_{1}+x_{2}\right)+x_{2}
$$
subject to the nonlinear constraints
$$
\begin{aligned}
   x_{0} x_{1} x_{2} x_{3} &\geq 25 \\
   x_{0}^{2}+x_{1}^{2}+x_{2}^{2}+x_{3}^{2} & = 40
\end{aligned}
$$
and the variable bounds
$$
1 \leq x_{i} \leq 5, \quad i=0, 1, 2, 3
$$
The initial guess is given by
$$
\boldsymbol{x} = [1,5,5,1]^T
$$

(Note that we have used 0-based indexing in the problem statement, whereas Hock and Schittkowski used 1-based indexing.)

## YAPSS Solution

The first step is to define the shape of the problem. In this case, the problem is static, not dynamic (there's no time-like independent variable), and so there are no phases, which is said by leaving the `phases=` keyword out: zero is a count like any other. There are four elements of the vector $\boldsymbol{x}$ which are the decision variables, so $\boldsymbol{x}$ is the parameter vector. There are also two constraints, one an inequality and one an equality. The class definitions are then

In [ ]:
import yapss


class Parameter(yapss.Parameter):
    x = yapss.vector(4)  # decision variables


class Discrete(yapss.Discrete):
    product = yapss.scalar()  # the product of all four, at least 25
    sum_of_squares = yapss.scalar()  # the sum of their squares, exactly 40

Note that we named individual scalar attributes of the `Discrete` class, but there is only a single named vector attribute of the `Parameter` class. That decision reflects the structure of the problem: there's not much to distinguish the elements of $\boldsymbol{x}$ other than their indices, but the two constraints are told apart by their names, making it much easier to connect the discrete constraint name with its formula.

Given the class definitions, we can now instantiate the problem:

In [ ]:
problem = yapss.Problem("HS071", parameter=Parameter, discrete=Discrete)

Then define the objective and discrete constraint callback functions:

In [ ]:
@problem.register.objective
def objective(arg):
    x = arg.parameter.x
    return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]


@problem.register.discrete
def discrete(arg, out):
    x = arg.parameter.x
    out.discrete.product = x[0] * x[1] * x[2] * x[3]
    out.discrete.sum_of_squares = x[0] ** 2 + x[1] ** 2 + x[2] ** 2 + x[3] ** 2
    return out

Set the bounds on the parameters and the constraint functions, per the problem statement:

In [ ]:
problem.parameter.x.bounds[:] = (1.0, 5.0)
problem.discrete.product.bounds = (25.0, None)
problem.discrete.sum_of_squares.bounds = (40.0, 40.0)

We also provide an initial guess for the parameter values:

In [ ]:
problem.parameter.x.guess[:] = [1.0, 5.0, 5.0, 1.0]

Specify the YAPSS and Ipopt options:

In [ ]:
problem.derivatives.method = "auto"
problem.derivatives.order = "second"
problem.ipopt_options.print_user_options = "no"
problem.ipopt_options.print_level = 3

Solve the problem:

In [ ]:
solution = problem.solve()

Everything the problem declares is read back under its own names: the parameters as `solution.parameter.x`, the constraint values as `solution.discrete.product` and `solution.discrete.sum_of_squares`, their multipliers as `solution.multiplier.discrete.*`, and the multipliers of the parameter bounds as `solution.multiplier.parameter.x` — one number per parameter, zero unless a bound is active. That is what a YAPSS program should read.

The last block reaches into `solution.nlp` for one reason: the Ipopt C++ example prints the bound multipliers as Ipopt holds them, `z_L` and `z_U` separately, and we want to compare like with like. `solution.nlp` is the record of what Ipopt was given and returned, in Ipopt's own order — an order YAPSS does not promise and may change. Here the four parameters happen to be the four NLP variables, one for one, but nothing entitles you to assume that: `solution.nlp.index` decodes the layout, and `index.variable.parameter.x` gives the positions to read.

In [ ]:
print("Solution of the primal variables, x")
for i, value in enumerate(solution.parameter.x):
    print(f"x[{i}] = {value:1.6e}")

print("\nMultipliers of the parameter bounds")
for i, value in enumerate(solution.multiplier.parameter.x):
    print(f"x[{i}] = {value:1.6e}")

print("\nSolution of the constraint multipliers, lambda")
print(f"lambda[product]        = {solution.multiplier.discrete.product:1.6e}")
print(f"lambda[sum_of_squares] = {solution.multiplier.discrete.sum_of_squares:1.6e}")

print("\nConstraint values")
print(f"product        = {solution.discrete.product:1.6e}")
print(f"sum of squares = {solution.discrete.sum_of_squares:1.6e}")

print("\nObjective value")
print(f"f(x*) = {solution.objective:1.6e}")

# Ipopt's own view of the same bound multipliers, to compare with the C++ example
print("\nBound multipliers as Ipopt holds them, z_L and z_U")
nlp = solution.nlp
for i, position in enumerate(nlp.index.variable.parameter.x):
    print(
        f"z_L[{i}] = {nlp.mult_x_L[position]:1.6e}\tz_U[{i}] = "
        f"{nlp.mult_x_U[position]:1.6e}"
    )

The solution above replicates the example given (in C++) in the [Ipopt interface documentation](https://coin-or.github.io/Ipopt/INTERFACES.html#INTERFACE_CPP), in much less code: the same variables, the same multipliers, and the same objective, to every digit printed there. That is the point of the comparison — YAPSS hands Ipopt the same problem — and it is the only reason this page reads anything out of `solution.nlp`.

## References